# 🐱 シグマゼミ 第1回 公式実験教材
## 「ChatGPTは次の言葉を当てるだけ？ — 自作の“次の文字当てAI”で確かめてみた」

[▶️ シグマゼミ YouTubeチャンネルはこちら](https://www.youtube.com/@sigmazemi)

---

### 🔰 はじめに（スマホ・PCで1秒で動かす方法）
1. 下の各コードブロックの左上にある **「▶️（再生ボタン）」** を上から順にポチポチ押すだけです！
2. 初回実行時に「Google作成ではありません」という警告が出ますが、外部ライブラリを一切使わない安全なコードですので **「このまま実行」** を押してください。
3. 面白い文章が生成されたら、ぜひ動画のコメント欄で教えてください！

※ 高校「情報I」の探究学習や、大学のプログラミング・情報科学のレポート等への引用・参考はご自由にどうぞ。

### 📥 ステップ1：夏目漱石『吾輩は猫である』（30万文字）を準備する
青空文庫から自動でダウンロードして、ルビや空白を取り除きます。（約1秒で完了します）

In [ ]:
import urllib.request
import zipfile
import re
import os

# 青空文庫『吾輩は猫である』のZIP URL
AOZORA_URL = "https://www.aozora.gr.jp/cards/000148/files/789_ruby_5639.zip"

print("[*] 青空文庫からダウンロード中...")
req = urllib.request.Request(AOZORA_URL, headers={"User-Agent": "Mozilla/5.0"})
with urllib.request.urlopen(req) as resp, open("wagahai.zip", "wb") as f:
    f.write(resp.read())

with zipfile.ZipFile("wagahai.zip", "r") as zf:
    for name in zf.namelist():
        if name.endswith(".txt"):
            with zf.open(name) as src, open("wagahai.txt", "wb") as dst:
                dst.write(src.read())
            break

raw_text = open("wagahai.txt", encoding="shift_jis", errors="ignore").read()
# ルビや注記を削除してきれいな日本語テキストにする
body = raw_text.split("-------------------------------------------------------")[-1].split("底本：")[0]
clean_text = re.sub(r"《.*?》|［＃.*?］|｜|\s+", "", body)

print(f"[+] 準備完了！ 総文字数: {len(clean_text):,} 文字")
print(f"    冒頭部分: {clean_text[:30]}...")

### 🧠 ステップ2：直前の文字数（コンテキスト）による文章の変化を体感する
- **0文字（記憶なし）**：文字の出やすさだけで選ぶので、ぐちゃぐちゃな文字の羅列
- **1文字（直前1文字）**：日本語っぽいつながりは出るが、意味は通らない
- **4文字（直前4文字）**：漱石っぽい文章になる。……ただし実は原作の切れ端をつないでいるだけ（ステップ4で確かめます）

In [ ]:
import random
from collections import Counter, defaultdict

# 乱数シードを固定（何度実行しても同じ結果が再現されます）
random.seed(42)

def train_model(text, n):
    """直前 n 文字から次の1文字の出現回数を集計する"""
    table = defaultdict(Counter)
    for i in range(len(text) - n):
        context = text[i:i + n]
        next_char = text[i + n]
        table[context][next_char] += 1
    return table

def generate_text(table, n, seed_prompt, length=100):
    """次の文字の確率サイコロを振って文章を生成する"""
    current = seed_prompt[-n:] if n > 0 else ""
    result = list(seed_prompt)
    for _ in range(length):
        counter = table[current]
        if not counter:
            break
        chars, weights = zip(*counter.items())
        chosen = random.choices(chars, weights=weights)[0]
        result.append(chosen)
        current = (current + chosen)[-n:] if n > 0 else ""
    return "".join(result)

print("=== [0文字AI] 記憶なし（文字の出やすさだけでサイコロを振る） ===")
model_0 = train_model(clean_text, 0)
print(generate_text(model_0, 0, "吾輩は", length=60))
print()

print("=== [1文字AI] 直前1文字だけ見てサイコロを振る ===")
model_1 = train_model(clean_text, 1)
print(generate_text(model_1, 1, "吾輩は", length=60))
print()

print("=== [4文字AI] 直前4文字を見てサイコロを振る（漱石っぽくなる？） ===")
model_4 = train_model(clean_text, 4)
print(generate_text(model_4, 4, "吾輩は猫で", length=120))


### 🎲 ステップ3：AIの頭の中を覗く（確率サイコロ）
「吾輩は猫で」と入力されたとき、AIは頭の中でどんなサイコロを振っているのでしょうか？
次の文字の確率ランキングTOP 5を表示します。

In [ ]:
context = "猫で"
counter = train_model(clean_text, 2)[context]
total = sum(counter.values())

print(f"【「{context}」の直後に来る文字の確率ランキング】（全 {total} 回観測）")
for char, count in counter.most_common(5):
    prob = (count / total) * 100
    bar = "█" * int(prob / 2)
    print(f"  「{char}」 : {prob:5.1f}% ({count:3d}回) {bar}")

print("\n➔ 『あ（ある）』の確率が 43.3% でダントツ！ この小さなAIは意味を考えているのではなく、数えた回数（確率）に従って選んでいるだけです。")

### ⚠️ ステップ4：記憶を増やしすぎるとどうなる？（過学習の罠）
記憶する文字数をどんどん増やすと、学習した文章では完璧になりますが、**「初めて見る未知の文章」**に出会った瞬間に大パニック（びっくりポイント急上昇）を起こします。
これが現代の最新AIでも大問題になる **「過学習（Overfitting）」** です。

In [ ]:
import math

# 動画と同じ条件：前の90%で学習し、最後の10%（先頭20,000文字）を「初めて見る文章」にする
split_idx = int(len(clean_text) * 0.9)
train_data = clean_text[:split_idx]
test_data = clean_text[split_idx:split_idx + 20000]
vocab = len(set(clean_text))

def calc_surprise(table, data, k):
    """直前 k 文字を見るAIの、1文字あたりの平均びっくりポイント（ビット）"""
    bits = 0.0
    for i in range(k, len(data)):
        ctx = data[i - k:i]
        nxt = data[i]
        c = table[ctx][nxt]
        tot = sum(table[ctx].values())
        # 見たことのない並びでも確率0にならないよう、ごく小さな値を足す（平滑化）
        p = (c + 0.0001) / (tot + 0.0001 * vocab)
        bits += -math.log2(p)
    return bits / (len(data) - k)

print("直前文字数 | 学習に使った文章 | 初めて見る文章")
print("-------------------------------------------------")
for k in [0, 1, 2, 3, 4]:
    table = train_model(train_data, k)
    tr_score = calc_surprise(table, train_data[:20000], k)
    te_score = calc_surprise(table, test_data, k)
    print(f"   {k} 文字   |     {tr_score:5.2f} ビット    |    {te_score:5.2f} ビット {'⚠️ 丸暗記' if te_score > tr_score + 2 else ''}")

print("\n➔ 学習に使った文章では 7.68 → 0.98 と下がるのに、初めて見る文章では 6.30 から上がっていきます（動画のグラフと同じ数字）。")


---

### 🎯 自分のレベルに合わせて遊んでみよう！（自由課題）

#### 【Lv.1（初級）】パラメータを変えてみる
- ステップ2のコードで、`length=120` を `200` や `300` に変えて、もっと長い文章を生成させてみよう！
- `吾輩は猫で` の部分を、`吾輩は犬で` や `吾輩は人間で` に変えるとどうなるかな？

#### 【Lv.2（中級）】別の小説でAIを作ってみる
- ステップ1の `AOZORA_URL` を、太宰治の『走れメロス』に変えてみよう！
  - 太宰治『走れメロス』ZIP URL: `https://www.aozora.gr.jp/cards/000035/files/1567_ruby_4948.zip`
- メロスAIがどんな劇的な文章を生成したか、ぜひコメント欄で教えてね！

#### 【Lv.3（上級・ガチ勢向け）】過学習を防ぐアルゴリズムを考えてみよう
- 直前4文字で見つからないときは直前3文字、それも見つからないときは直前2文字にフォールバックする「バックオフ（Katz Backoff）」や、より高度な平滑化（Good-Turingなど）を実装してみよう！

---
**シグマゼミ【AIのなぜをずんだもん解説】**
- YouTube: [@sigmazemi](https://www.youtube.com/@sigmazemi)
- 出典: Shannon (1948), Bengio et al. (2003), 青空文庫